In [ ]:
# pip install -U ddgs

In [1]:
from langchain_google_genai import GoogleGenerativeAI
from dotenv import load_dotenv
import os
import time

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

model = GoogleGenerativeAI(model="gemini-3.1-flash-lite")
model.invoke("최근 로제가 발표한 신곡은 무엇인가요?")

"로제가 최근 발표한 신곡은 브루노 마스(Bruno Mars)와 함께한 협업곡 **'APT.' (아파트)**입니다.\n\n이 곡은 10월 18일에 발매되었으며, 한국의 술자리 게임인 '아파트 게임'에서 착안한 중독성 있는 후렴구로 전 세계적으로 큰 인기를 끌고 있습니다."

In [2]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults(results_seperator=";\n")
docs = search.invoke("최근 로제가 발표한 신곡은 무엇인가요?")

print(docs)

C:\Users\bny64\AppData\Local\Temp\ipykernel_11788\592159615.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


snippet: 지난해 10월 로제가 세계적 가수 브루노 마스와 함께 발표한 곡 'APT.'(아파트)가 최근 빌보드 메인 차트인 '핫 100'에서 2주 연속 3위를 차지하며 대중적 인기를 과시하고 있기 때문이다., title: 2년간 K팝 잊은 그래미…내년 로제가 자존심 회복시킬까 [N초점] - 뉴스1, link: https://www.news1.kr/entertain/music/5682705, snippet: [사진 = ROSÉ & Bruno Mars - APT. (Official MV)]. [이코노미 트리뷴 = 김용현 기자] 최근 국내에서 이른바 ‘로제’ 바람이 거세게 불고 있다. 4인조 걸그룹 ‘블랙핑크’ 메인보컬 로제가 최근 발간한 싱글 앨범 수록곡 ‘아파트(APT) 얘기다., title: 블랙핑크 로제가 쏘아올린 ‘한국판 스위프트노믹스’ | 이코노미 트리뷴, link: https://economytribune.co.kr/View.aspx?No=3422593, snippet: 로제가 지난해 10월 발표한 ‘아파트’ 역시 여전히 차트를 지켜, 솔로곡 총 2곡을 나란히 차트에 올렸다., title: [스경X이슈] “‘아파트’ 133억 수익” 로제, ‘메시’도 英 차트 진입, link: https://sports.khan.co.kr/article/202505171019003, snippet: 걸그룹 블랙핑크 멤버 로제가 발표한 신곡 ‘아파트(APT.)’의 글로벌 인기에 덩달아 국내 주류 업체인 하이트진로의 주가가 24일 6% 급등했다., title: 로제 신곡 ‘아파트’ 덕분… 하이트진로 6% 급등 | 조선일보, link: https://www.chosun.com/economy/money/2024/10/25/EKPTTE7YFZGADLUTUG644DSPKM/


In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "아래 context에 기반하여 사용자의 질문에 답변하라.:\n\n{context}"),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = question_answering_prompt | model

In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory

# 채팅 메시지를 저장할 메모리 객체 생성
chat_history = ChatMessageHistory()

# 사용자 질문을 메모리에 저장
chat_history.add_user_message("요즘 로제가 발표한 신곡은 무엇인가요?")

# 문서 검색하고 답변 생성
answer = document_chain.invoke({"messages": chat_history.messages, "context": docs})

# 생성된 답변을 메모리에 저장
chat_history.add_ai_message(answer)
print(answer)

요즘 로제가 발표하여 큰 인기를 얻고 있는 신곡은 브루노 마스와 함께 부른 **'APT.'(아파트)**입니다.


In [5]:
# DuckDuckGo API 래퍼를 사용하여 검색할 때 검색 매개변수를 설정하는 클래스 import
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# 한국 지역("kr-kr")을 기준, 최근 일주일("w") 내의 검색 결과를 가져오도록 초기화
wrapper = DuckDuckGoSearchAPIWrapper(region="kr-kr", time="w")

In [6]:
# 검색 기능을 위한 DuckDuckGoSearchResults 초기화
search = DuckDuckGoSearchResults(
    api_wrapper=wrapper,  # 앞에서 정의한 API 래퍼를 사용
    source="news",  # 뉴스 소스에서만 검색하도록 지정
    results_separator=";\n",  # 결과 항목 사이에 구분자 사용(세미콜론과 줄 바꿈)
)

In [8]:
# "로제의 신곡 APT에 대한 반응"을 검색하고 결과를 docs에 저장
docs = search.invoke("로제의 신곡 APT에 대한 반응")

# 검색 결과 출력
print(docs)

snippet: 3 days ago - "APT." is a song by New Zealand and South Korean singer Rosé and American singer-songwriter Bruno Mars. It was released through The Black Label and Atlantic Records on 18 October 2024, as the lead single from Rosé's debut studio album, Rosie (2024). "APT." marked Rosé's first solo single ..., title: APT. (song) - Wikipedia, link: https://en.wikipedia.org/wiki/APT._(song);
snippet: 1 week ago - 로제의 이 독창적인 그루브와 목소리는 테디가 작곡한 블랙핑크 곡들이나 팝송과 만나면 시너지가 좋은 편인데, 이것은 로제의 보컬이 스탠다드한 기본기 탄탄한 발성보다 YG엔터테인..., title: 로제(BLACKPINK) - 나무위키, link: https://namu.wiki/w/로제(BLACKPINK);
snippet: 1 week ago - [서울=뉴시스]박영환 기자 = 로제와 브루노 마스의 히트곡 ‘아파트(APT.)’를 비롯해 세계 각국에서 발표된 약 500곡에서 서로 비슷한 네 음짜리 선율이 발견됐다는 분석이 나왔다. 덴마크 음악가 칼 마틴(27)이 수개월간 곡을 ..., title: 로제 '아파트' 속 그 4개 음…세계 각국 72곡서 닮은 선율 찾았다 :: 공감언론 뉴시스 ::, link: https://www.newsis.com/view/NISX20260728_0003726426;
snippet: 1 day ago - BLACKPINK’s Rosé and Bruno Mars’ “APT.” continues its record-breaking run on YouTube! On the morning of July 31

In [9]:
# DuckDuckGo를 이용해 ytn.co.kr 웹 사이트에서 로제의 신곡 APT에 대한 분석을 검색
docs = search.invoke("site:ytn.co.kr 로제의 신곡 APT에 대한 분석")

docs

'snippet: 5 days ago - K팝의 경우 올해 2월 제68회 \'그래미 어워즈\'에선 지난해 최고 히트곡인 \'케이팝 데몬 헌터스\' 오리지널사운드트랙 \'골든\'(GOLDEN)이 OST 부문에 해당하는 \'베스트 송 리튼 포 비주얼 미디어\' 1개 부문만 수상하고, 이에 못지않게 흥행한 로제의 \'아파트\'(APT.)가 무관에 그치기도 했습니다., title: [지금이뉴스]"출품 거부" BTS 멤버 전원, 충격적 선언…보수적 시상식에 \'일침\' | YTN, link: https://www.ytn.co.kr/_ln/0545_202607300800018912;\nsnippet: 5 days ago - ■ 방송 : YTN 라디오 FM 94.5 (09:00~10:00)■ 진행 : 이현웅 아나운서 대타 진행■ 방송일 : 2026년 7월 29일 수요일■ 전화 : 박승진 하나증권 리서치센터 해외주식분석실..., title: [경제]레버리지ETF, 70대가 가장 \'큰 손\' "추가 매수도 손절도 어려운 구간 진입" | YTN, link: https://www.ytn.co.kr/_ln/0102_202607291031105970;\nsnippet: 대한민국 24시간 뉴스 방송채널, 실시간 속보 및 제보하기, 분야별 뉴스, 방송 프로그램 다시보기, title: 한국의 뉴스 채널 YTN (채널24), link: https://ytn.co.kr/;\nsnippet: 3 days ago - 가수 싸이가 온라인상의 악플에 직접 댓글을 달아 화제다.싸이는 지난 30일 자신의 SNS에 ‘싸이 흠뻑쇼 서머스웨그 2026’(이하 \'흠뻑쇼 2026\') 관련 영상을 올렸다. 공개..., title: [가요] "\'흠뻑쇼\' 맨날 같은 노래" 악플에…싸이, 직접 등판 "와보세요" | YTN, link: https://star.ytn.co.kr/_sn/0117_202607311439132659'

In [ ]:
# 검색 결과의 링크들을 저장할 빈 리스트 초기화
links = []

# 검색 결과를 세미콜론과 줄 바꿈 기준으로 분리하고, 각 결과 항목에서 링크 추출
for doc in docs.split(";\n"):
    print(doc)  # 각 검색 결과 항목을 출력하여 확인
    link = doc.split("link:")[1].strip()  # 각 항목에서 'link:' 이후의 URL 부분만 추출
    links.append(link)  # 추출한 링크를 리스트에 추가

# 모든 링크를 출력
print(links)

snippet: 5 days ago - K팝의 경우 올해 2월 제68회 '그래미 어워즈'에선 지난해 최고 히트곡인 '케이팝 데몬 헌터스' 오리지널사운드트랙 '골든'(GOLDEN)이 OST 부문에 해당하는 '베스트 송 리튼 포 비주얼 미디어' 1개 부문만 수상하고, 이에 못지않게 흥행한 로제의 '아파트'(APT.)가 무관에 그치기도 했습니다., title: [지금이뉴스]"출품 거부" BTS 멤버 전원, 충격적 선언…보수적 시상식에 '일침' | YTN, link: https://www.ytn.co.kr/_ln/0545_202607300800018912
snippet: 5 days ago - ■ 방송 : YTN 라디오 FM 94.5 (09:00~10:00)■ 진행 : 이현웅 아나운서 대타 진행■ 방송일 : 2026년 7월 29일 수요일■ 전화 : 박승진 하나증권 리서치센터 해외주식분석실..., title: [경제]레버리지ETF, 70대가 가장 '큰 손' "추가 매수도 손절도 어려운 구간 진입" | YTN, link: https://www.ytn.co.kr/_ln/0102_202607291031105970
snippet: 대한민국 24시간 뉴스 방송채널, 실시간 속보 및 제보하기, 분야별 뉴스, 방송 프로그램 다시보기, title: 한국의 뉴스 채널 YTN (채널24), link: https://ytn.co.kr/
snippet: 3 days ago - 가수 싸이가 온라인상의 악플에 직접 댓글을 달아 화제다.싸이는 지난 30일 자신의 SNS에 ‘싸이 흠뻑쇼 서머스웨그 2026’(이하 '흠뻑쇼 2026') 관련 영상을 올렸다. 공개..., title: [가요] "'흠뻑쇼' 맨날 같은 노래" 악플에…싸이, 직접 등판 "와보세요" | YTN, link: https://star.ytn.co.kr/_sn/0117_202607311439132659
['https://www.ytn.co.kr/_ln/0545_202607300800018912', 'https://www.ytn.co

In [13]:
# 랭체인의 WebBaseLoader를 사용하여 웹 페이지의 내용 불러오기
from langchain_community.document_loaders import WebBaseLoader

# WebBaseLoader 객체를 생성. 'links'는 웹 페이지의 URL 목록을 담고 있는 변수
# bs_get_text_kwargs는 BeautifulSoup의 get_text() 메서드에 전달될 추가 인자
loader = WebBaseLoader(
    web_paths=links,  # 웹 페이지의 링크 목록을 지정
    bs_get_text_kwargs={
        "strip": True
    },  # 웹 페이지에서 텍스트를 가져올 때 앞뒤의 공백 제거
)

# 비동기로 웹 페이지의 내용을 로드하고, 각 문서를 page_contents 리스트에 추가
page_contents = []  # 각 웹 페이지의 내용을 저장할 리스트
async for doc in loader.alazy_load():
    page_contents.append(doc) #불러온 문서를 page_contents 리스트에 추가

# page_contents에 있는 각 웹 페이지의 내용 출력
for content in page_contents:
    print(content) # 웹 페이지의 내용 출력
    print('--------------------')

Fetching pages: 100%|##########| 4/4 [00:00<00:00,  4.46it/s]


page_content='[지금이뉴스]"출품 거부" BTS 멤버 전원, 충격적 선언…보수적 시상식에 '일침'  | YTN메뉴 바로가기본문 바로가기푸터 바로가기닫기YTNYTN브랜드채널YTN 회사소개INSIDE YTNYTN 사이언스YTN 라디오YTN2YTN dmbYTN world남산서울타워장애인 서비스제보LIVE로그인회원가입로그아웃회원정보변경정치경제사회전국국제문화스포츠연예비즈날씨이슈시리즈TV프로그램"출품 거부" BTS 멤버 전원, 충격적 선언…보수적 시상식에 '일침' [지금이뉴스]검색검색하기닫기많이 본 뉴스LIVE공유공유하기닫기페이스북엑스밴드카카오톡복사하기전체메뉴YTN닫기홈최신뉴스LIVE제보하기뉴스정치과학경제문화사회스포츠전국연예국제비즈날씨게임이슈속보단독재난시리즈몇층이세요한방이슈짤막상식와이즈픽자막뉴스뉴스모아제보영상와이파일운세앵커리포트나이트포커스지금이뉴스이게웬날리지Y녹취록이슈톺TV프로그램프로그램별날짜별앵커 소개시청자의견장애인 서비스검색하기"출품 거부" BTS 멤버 전원, 충격적 선언…보수적 시상식에 '일침' [지금이뉴스]2026.07.30. 오전 08:00.댓글글자크기설정글자 크기 설정닫기가가가가가공유하기공유하기닫기페이스북엑스밴드카카오톡복사하기인쇄하기AD그룹 방탄소년단(BTS)이 내년 2월 열리는 제69회 '그래미 어워즈'에 음악을 출품하지 않는다고 전격 선언했습니다.29일 오후 방탄소년단 멤버들은 일제히 소셜미디어(SNS)를 통해 "저희는 올해 그래미에 출품하지 않기로 했다"고 밝히며 "음악이 지역이나 언어로 구분되기보다 음악 그 자체로 들리고 사랑받을 수 있기를 바란다"고 전했습니다.앞서 그래미 어워즈 측은 지난달 한국·중국·일본 등 아시아권 음악을 구분해 '베스트 아시안 팝 뮤직 퍼포먼스' 부문을 신설한다고 밝힌 바 있습니다.'베스트 아시안 팝 뮤직 퍼포먼스'는 K팝을 비롯해 J팝(일본 팝음악)과 C팝(중국 팝음악) 등 하나 또는 그 이상의 아시안 국가 언어를 유의미하게 사용하고, 아시아 출신이거나 현지에서 인정받는 아시안 팝 음악의 탁월함을 기리기 위해 만들어